# 身份验证
`Starlette`中的身份验证功能是通过`AuthenticationMiddleware`来提供的。`AuthenticationMiddleware`在配置是需要通过参数`backend`提供一个继承了`Authenticationbackend`基类的验证类。验证的结果会被保存在`request.user`和`request.auth`中。

继承`AuthenticationMiddleware`类主要需要实现其中的`async def authenticate(self, request) -> AuthCredentials`, `User`方法。该方法返回一个元组，第一个元素是验证结果标记，第二个元素是用户信息。

`AuthCredentials`类通常用于存放验证结果标记，包括是否通过验证以及权限组合等。这些标记可以在`Endpoint`处使用`@require()`修饰器进行过滤。`@require()`修饰器除了可以接受一个字符串作为参数，还可以接受一个字符串数组作为参数，来进行标记的组合。默认情况下，验证不通过将会返回403错误，但是`@require()`可以通过`status_code`参数重新指定验证不通过时的错误号。

验证过程中抛出的异常，可以通过`add_middleware()`中的参数`on_error`来捕获处理。



In [ ]:
from starlette.applications import Starlette
from starlette.authentication import (
    AuthCredentials, AuthenticationBackend, AuthenticationError, SimpleUser
)
from starlette.middleware import Middleware
from starlette.middleware.authentication import AuthenticationMiddleware
from starlette.responses import PlainTextResponse
from starlette.routing import Route
import base64
import binascii


class BasicAuthBackend(AuthenticationBackend):
    async def authenticate(self, conn):
        if "Authorization" not in conn.headers:
            return

        auth = conn.headers["Authorization"]
        try:
            scheme, credentials = auth.split()
            if scheme.lower() != 'basic':
                return
            decoded = base64.b64decode(credentials).decode("ascii")
        except (ValueError, UnicodeDecodeError, binascii.Error) as exc:
            raise AuthenticationError('Invalid basic auth credentials')

        username, _, password = decoded.partition(":")
        # TODO: You'd want to verify the username and password here.
        return AuthCredentials(["authenticated"]), SimpleUser(username)


async def homepage(request):
    if request.user.is_authenticated:
        return PlainTextResponse('Hello, ' + request.user.display_name)
    return PlainTextResponse('Hello, you')

routes = [
    Route("/", endpoint=homepage)
]

middleware = [
    Middleware(AuthenticationMiddleware, backend=BasicAuthBackend())
]

app = Starlette(routes=routes, middleware=middleware)

## 用户
一旦`AuthenticationMiddleware`安装，该`request.user`接口将可供端点或其他中间件使用。

该接口应该是子类`BaseUser`，它提供两个属性，以及用户模型包含的任何其他信息。

- `.is_authenticated`
- `.display_name`

`Starlette` 提供两种内置用户实现：`UnauthenticatedUser()`和`SimpleUser(username)`。

## 授权凭证
重要的是，身份验证凭证应被视为与用户不同的概念。身份验证方案应该能够独立于用户身份来限制或授予特定权限。

该类`AuthCredentials`提供了公开的基本接口`request.auth `：

- `.scopes`

## 权限
权限作为端点装饰器实现，强制传入的请求包含所需的身份验证范围。

In [ ]:
from starlette.authentication import requires


@requires('authenticated')
async def dashboard(request):
    ...

您可以包含一个或多个必需范围：

In [ ]:
from starlette.authentication import requires


@requires(['authenticated', 'admin'])
async def dashboard(request):
    ...

默认情况下，未授予权限时将返回 403 响应。在某些情况下，您可能需要自定义此设置，例如，对未经身份验证的用户隐藏有关 URL 布局的信息。

In [ ]:
from starlette.authentication import requires


@requires(['authenticated', 'admin'], status_code=404)
async def dashboard(request):
    ...

或者，您可能希望将未经身份验证的用户重定向到其他页面。

In [ ]:
from starlette.authentication import requires


async def homepage(request):
    ...


@requires('authenticated', redirect='homepage')
async def dashboard(request):
    ...

当重定向用户时，您将他们重定向到的页面将包含他们在next查询参数中最初请求的 URL：

In [ ]:
from starlette.authentication import requires
from starlette.responses import RedirectResponse


@requires('authenticated', redirect='login')
async def admin(request):
    ...


async def login(request):
    if request.method == "POST":
        # Now that the user is authenticated,
        # we can send them to their original request destination
        if request.user.is_authenticated:
            next_url = request.query_params.get("next")
            if next_url:
                return RedirectResponse(next_url)
            return RedirectResponse("/")

对于基于类的端点，您应该将装饰器包装在类的方法周围。

In [ ]:
from starlette.authentication import requires
from starlette.endpoints import HTTPEndpoint


class Dashboard(HTTPEndpoint):
    @requires("authenticated")
    async def get(self, request):
        ...

## 自定义身份验证错误响应

`AuthenticationError`您可以自定义当身份验证后端引发错误时发送的错误响应：

In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.authentication import AuthenticationMiddleware
from starlette.requests import Request
from starlette.responses import JSONResponse


def on_auth_error(request: Request, exc: Exception):
    return JSONResponse({"error": str(exc)}, status_code=401)

app = Starlette(
    middleware=[
        Middleware(AuthenticationMiddleware, backend=BasicAuthBackend(), on_error=on_auth_error),
    ],
)